In [1]:
import pandas as pd
import json
import re
import unicodedata
import math
from collections import Counter

# Change the paths only if your files are in another folder
test = pd.read_csv("test_questions.csv")

with open("retrievals.json", "r", encoding="utf-8") as f:
    retrievals = json.load(f)

print("Number of test questions:", len(test))
print("Number of retrieval lists:", len(retrievals))
print("Retrieved passages per question:", len(retrievals[0]))

print("\nTest columns:")
print(test.columns.tolist())

test.head()

Number of test questions: 248
Number of retrieval lists: 248
Retrieved passages per question: 5

Test columns:
['id', 'domain', 'question', 'gold']


,id,domain,question,gold
0,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...
1,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...
2,passport_216,passport,বিদেশে শ্রমিক শিক্ষার্থীর ৬৪ পৃষ্ঠা ৫ বছর এক্স...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...
3,passport_241,passport,স্টুডেন্ট হিসেবে মিশনে ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্র...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...
4,passport_291,passport,শিশুর পাসপোর্ট সংগ্রহে কী লাগবে?,অপ্রাপ্তবয়স্কের পাসপোর্ট সংগ্রহে পিতা বা মাতা...


In [2]:
def normalize(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = re.sub(r"[^\w\u0980-\u09FF]+", " ", text)
    return " ".join(text.split())


def token_coverage(reference, text):
    """
    How much of the reference appears in the text.
    Returns value between 0 and 1.
    """
    ref_tokens = Counter(normalize(reference).split())
    text_tokens = Counter(normalize(text).split())

    common = sum((ref_tokens & text_tokens).values())
    total = sum(ref_tokens.values())

    return common / total if total > 0 else 0


rows = []

for i in range(len(test)):

    question = test.loc[i, "question"]
    gold = test.loc[i, "gold"]

    for rank, item in enumerate(retrievals[i], start=1):

        context = item["text"]
        title = item["title"]

        # Does the passage contain the gold-answer information?
        gold_coverage = token_coverage(gold, context)

        # Is the passage also related to the question?
        question_coverage = token_coverage(
            question,
            title + " " + context
        )

        # Simple relevance judgement
        if gold_coverage >= 0.75 and question_coverage >= 0.30:
            grade = 2       # highly relevant
        elif gold_coverage >= 0.45 and question_coverage >= 0.30:
            grade = 1       # relevant
        else:
            grade = 0       # not relevant

        rows.append({
            "id": test.loc[i, "id"],
            "domain": test.loc[i, "domain"],
            "question": question,
            "gold": gold,
            "rank": rank,
            "doc_id": item["doc_id"],
            "title": title,
            "context": context,
            "gold_coverage": gold_coverage,
            "question_coverage": question_coverage,
            "grade": grade
        })


labels = pd.DataFrame(rows)

labels.head(10)

,id,domain,question,gold,rank,doc_id,title,context,gold_coverage,question_coverage,grade
0,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...,1,passport_kb_003,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ,ই-পাসপোর্ট আবেদনের সাধারণ পাঁচটি ধাপ হলো: প্রথ...,0.477273,0.750000,1
1,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...,2,passport_cov_001,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...,1.000000,1.000000,2
2,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...,3,passport_web_20260807_001,ই-পাসপোর্ট আবেদনের সরকারি পাঁচ ধাপ,সরকারি ই-পাসপোর্ট পোর্টাল অনুযায়ী আবেদন প্রক্র...,0.454545,0.625000,1
3,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...,4,passport_kb_001,ই-পাসপোর্ট সেবার সারসংক্ষেপ,"বাংলাদেশের ই-পাসপোর্ট সেবা মূলত অনলাইন আবেদন, ...",0.454545,0.250000,0
4,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...,5,passport_trainctx_048,অনলাইনে পাসপোর্ট আবেদন করলে পরে কী কী করতে হয়,ই-পাসপোর্টের সাধারণ প্রক্রিয়া হলো: প্রথমে আপন...,0.409091,0.500000,0
5,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,1,passport_cov_002,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,1.000000,1.000000,2
6,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,2,passport_trainctx_065,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর রেগুলার ই-পাস...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,0.904762,0.916667,2
7,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,3,passport_trainctx_017,বাংলাদেশ মিশনে ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ফি কত,বিদেশস্থ বাংলাদেশ মিশনে সাধারণ আবেদনকারীর জন্য...,0.857143,0.666667,2
8,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,4,passport_web_20260807_004,বিদেশস্থ বাংলাদেশ মিশনে ৬৪ পৃষ্ঠার ৫ বছর মেয়াদ...,বিদেশস্থ বাংলাদেশ মিশনে ৬৪ পৃষ্ঠা ও ৫ বছর মেয়া...,0.571429,0.416667,1
9,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,5,passport_trainctx_045,বাংলাদেশ মিশনে ৬৪ পৃষ্ঠা ৫ বছর রেগুলার ফি কত,বিদেশস্থ বাংলাদেশ মিশনে সাধারণ আবেদনকারীর জন্য...,0.761905,0.583333,2


In [3]:
# Convert grade into simple binary relevance
labels["relevant"] = (labels["grade"] > 0).astype(int)

# Show a few examples
pd.set_option("display.max_colwidth", 100)

display(
    labels[
        [
            "id",
            "question",
            "rank",
            "title",
            "gold_coverage",
            "question_coverage",
            "grade"
        ]
    ].head(20)
)

# Save the labels so you can manually inspect/edit them
labels.to_csv(
    "retrieval_relevance_labels.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: retrieval_relevance_labels.csv")

,id,question,rank,title,gold_coverage,question_coverage,grade
0,passport_001,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,1,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ,0.477273,0.750000,1
1,passport_001,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,2,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী,1.000000,1.000000,2
2,passport_001,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,3,ই-পাসপোর্ট আবেদনের সরকারি পাঁচ ধাপ,0.454545,0.625000,1
3,passport_001,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,4,ই-পাসপোর্ট সেবার সারসংক্ষেপ,0.454545,0.250000,0
4,passport_001,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,5,অনলাইনে পাসপোর্ট আবেদন করলে পরে কী কী করতে হয়,0.409091,0.500000,0
5,passport_179,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-পাসপোর্টে কত ডলার লাগে?,1,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-পাসপোর্টে কত ডলার লাগে,1.000000,1.000000,2
6,passport_179,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-পাসপোর্টে কত ডলার লাগে?,2,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর রেগুলার ই-পাসপোর্টে কত ডলার লাগে,0.904762,0.916667,2
7,passport_179,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-পাসপোর্টে কত ডলার লাগে?,3,বাংলাদেশ মিশনে ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ফি কত,0.857143,0.666667,2
8,passport_179,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-পাসপোর্টে কত ডলার লাগে?,4,বিদেশস্থ বাংলাদেশ মিশনে ৬৪ পৃষ্ঠার ৫ বছর মেয়াদি ই-পাসপোর্ট ফি,0.571429,0.416667,1
9,passport_179,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-পাসপোর্টে কত ডলার লাগে?,5,বাংলাদেশ মিশনে ৬৪ পৃষ্ঠা ৫ বছর রেগুলার ফি কত,0.761905,0.583333,2


Saved: retrieval_relevance_labels.csv


In [4]:
import numpy as np

# If you manually edited the CSV, reload it here:
# labels = pd.read_csv("retrieval_relevance_labels.csv")
# labels["relevant"] = (labels["grade"] > 0).astype(int)


def dcg(grades):
    score = 0

    for i, grade in enumerate(grades):
        score += (2**grade - 1) / math.log2(i + 2)

    return score


results = []

for qid, group in labels.groupby("id"):

    group = group.sort_values("rank")

    relevance = group["relevant"].tolist()
    grades = group["grade"].tolist()

    # Hit@k
    hit1 = int(any(relevance[:1]))
    hit3 = int(any(relevance[:3]))
    hit5 = int(any(relevance[:5]))

    # Reciprocal Rank
    rr = 0
    for rank, rel in enumerate(relevance, start=1):
        if rel == 1:
            rr = 1 / rank
            break

    # nDCG@5
    actual_dcg = dcg(grades)

    ideal_grades = sorted(grades, reverse=True)
    ideal_dcg = dcg(ideal_grades)

    ndcg5 = actual_dcg / ideal_dcg if ideal_dcg > 0 else 0

    results.append({
        "id": qid,
        "Hit@1": hit1,
        "Hit@3": hit3,
        "Hit@5": hit5,
        "RR": rr,
        "nDCG@5": ndcg5
    })


results = pd.DataFrame(results)

print("Hit@1 :", round(results["Hit@1"].mean(), 4))
print("Hit@3 :", round(results["Hit@3"].mean(), 4))
print("Hit@5 :", round(results["Hit@5"].mean(), 4))
print("MRR@5 :", round(results["RR"].mean(), 4))
print("nDCG@5:", round(results["nDCG@5"].mean(), 4))

Hit@1 : 0.8589
Hit@3 : 0.9718
Hit@5 : 0.9758
MRR@5 : 0.9116
nDCG@5: 0.9035
